In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv('XRF_databases/soil/plsda/soil.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1.32':'13.1']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1.32':'13.1'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1.32':'13.1'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

In [ ]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

## Definição das Zonas Espectrais

In [ ]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('Al', 1.33, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.19),
('S', 2.19, 2.55),
('Rh L + Ar', 2.55, 3.21),
('K', 3.21, 3.53),
('Ca ka', 3.53, 3.84),
('Ca kb', 3.84, 4.37),
('Ti ka', 4.37, 4.75),
('Ti kb', 4.75, 5.12),
('Cr', 5.12, 5.77),
('Mn', 5.77, 6.13),
('Fe ka', 6.13, 6.80),
('Fe kb', 6.80, 7.30),
('background1', 7.30, 7.91),
('Cu', 7.91, 8.20),
('background2', 8.20, 10.69),
('Fe ka + Ti ka', 10.69, 11.14),
('background3', 11.14, 12.55),
('sum Fe' , 12.55, 13.1)
]

## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [ ]:
def aggregate_spectral_zones_pca(spectral_zones_dict):
    """
    Agrega zonas espectrais usando PCA com 1 componente principal.
    
    Para cada zona espectral, ajusta uma PCA com 1 componente e extrai:
    - Scores: projeção das amostras na direção de máxima variância
    - Loadings: pesos de cada variável na PC1
    - Média: vetor de médias da zona (para reconstrução)
    - Variância Explicada: fração da variância capturada pela PC1
    
    Parameters
    ----------
    spectral_zones_dict : dict
        Dicionário retornado por extract_spectral_zones.
        Chaves = nomes das zonas, Valores = DataFrames com dados espectrais.
    
    Returns
    -------
    scores_df : pd.DataFrame
        DataFrame com scores da PC1 para cada zona (amostras x zonas).
    pca_info_dict : dict
        Dicionário com informações da PCA para cada zona:
        - 'loadings': vetor de loadings da PC1
        - 'mean': vetor de médias da zona
        - 'variance_explained': fração de variância explicada
        - 'columns': nomes das colunas originais (para reconstrução)
    """
    from sklearn.decomposition import PCA
    import pandas as pd

    scores_dict = {}  # armazena scores de cada zona
    pca_info_dict = {}  # armazena informações para reconstrução
    
    for zone_name, zone_df in spectral_zones_dict.items():
        # 1: Preparação dos dados
        X_zone = zone_df.values  # converter para numpy array
        
        # 2: Ajuste da PCA com 1 componente
        pca = PCA(n_components=1)
        scores = pca.fit_transform(X_zone)  # scores da PC1 (n_samples, 1)
        
        # 3: Extração das informações
        loadings = pca.components_[0]  # loadings da PC1 (d_m,)
        mean_vector = pca.mean_  # vetor de médias (d_m,)
        variance_explained = pca.explained_variance_ratio_[0]  # fração de variância
        
        # 4: Armazenamento
        scores_dict[zone_name] = scores.flatten()  # converter para 1D
        
        pca_info_dict[zone_name] = {
            'loadings': loadings,
            'mean': mean_vector,
            'variance_explained': variance_explained,
            'columns': zone_df.columns.tolist(),  # nomes das colunas originais
            'pca_model': pca  # modelo PCA completo (para uso futuro)
        }
        
        # Log informativo
        print(f"Zona '{zone_name}': VE = {variance_explained:.2%}, "
              f"dim = {len(loadings)} variáveis")
    
    # Criar DataFrame com todos os scores
    scores_df = pd.DataFrame(scores_dict)
    
    return scores_df, pca_info_dict

In [ ]:
def reconstruct_threshold_to_spectrum(threshold_value, zone_name, pca_info_dict):
    """
    Reconstrói um threshold escalar (no espaço dos scores) para o espaço 
    espectral original, gerando um "espectro de threshold" multivariado.
    
    Fórmula matemática:
        τ = mean + threshold_value * loadings
    
    Parameters
    ----------
    threshold_value : float
        Valor do threshold no espaço dos scores da PC1.
    zone_name : str
        Nome da zona espectral.
    pca_info_dict : dict
        Dicionário com informações da PCA (retornado por aggregate_spectral_zones_pca).
    
    Returns
    -------
    threshold_spectrum : pd.Series
        Espectro de threshold com índice = energias/comprimentos de onda originais.
    """
    import pandas as pd
    # Recuperar informações da PCA
    pca_info = pca_info_dict[zone_name]
    loadings = pca_info['loadings']
    mean_vector = pca_info['mean']
    columns = pca_info['columns']
    
    # Reconstrução: τ = mean + q * loadings
    threshold_spectrum = mean_vector + threshold_value * loadings
    
    # Converter para Series com índice original
    threshold_spectrum = pd.Series(threshold_spectrum, index=columns, name=f'threshold_{threshold_value:.4f}')
    
    return threshold_spectrum

In [ ]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

## Pipeline SMeX com Agregação por PCA

In [ ]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

In [ ]:
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred_cont
)

In [ ]:
def build_predicate_graphv2(bags_result, predicate_ranking_dict, 
                            metric_column='Cov',
                            random_state=42, show_details=True,
                            var_exp=False, pca_info_dict=None, predicates_df=None):
    """
    Constrói um grafo direcionado de predicados onde os pesos das arestas
    são baseados na Covriancia (ou outra métrica) do predicado de ORIGEM.
    
    Diferença da versão original (build_predicate_graph):
    - Versão original: peso = co-ocorrência entre predicados
    - Esta versão: peso = COV (ou métrica) do predicado de origem da aresta
    
    Parameters
    ----------
    - **bags_result** : dict
        Dicionário com bags de predicados:
        {'Bag_1': {'Ca ka <= 25.5': DataFrame, ...}, 'Bag_2': {...}, ...}
        
    - **predicate_ranking_dict** : dict
        Dicionário com rankings de predicados segundo uma métrica para cada bag:
        {'Bag_1': DataFrame(['Predicate', metric_column]), 'Bag_2': ...}
        
    - **metric_column** : str, default='Cov'
        Nome da coluna no predicate_ranking_dict que contém a métrica de ordenação.
        Permite flexibilidade para usar 'Cov', 'Permutation', etc.
        
    - **random_state** : int, default=42
        Semente para desempate aleatório de arestas bidirecionais.
        
    - **show_details** : bool, default=True
        Se True, imprime detalhes sobre remoção de arestas bidirecionais.
        
    - **var_exp** : bool, default=False
        Se True, multiplica os pesos das arestas pela variância explicada (PC1)
        da zona espectral correspondente ao predicado de origem.
        
    - **pca_info_dict** : dict, optional
        Dicionário com informações da PCA para cada zona (obrigatório se var_exp=True).
        Chaves = nomes das zonas, Valores = dict com 'variance_explained'.
        
    - **predicates_df** : pd.DataFrame, optional
        DataFrame com informações dos predicados (obrigatório se var_exp=True).
        Colunas obrigatórias: 'rule', 'zone'.
    
    Returns
    -------
    - **DG** : nx.DiGraph
        Grafo direcionado com pesos baseados na métrica acumulada.
    """
    import networkx as nx
    import numpy as np
    import pandas as pd
    
    # Validação dos parâmetros var_exp
    if var_exp:
        if pca_info_dict is None:
            raise ValueError("pca_info_dict é obrigatório quando var_exp=True")
        if predicates_df is None:
            raise ValueError("predicates_df é obrigatório quando var_exp=True")
        # Criar lookup: predicado (rule) → zona
        predicate_to_zone = dict(zip(predicates_df['rule'], predicates_df['zone']))
    
    # Define semente para reprodutibilidade nos desempates
    np.random.seed(random_state)
    
    # FASE 1: INICIALIZAÇÃO DO GRAFO
    # Cria grafo direcionado vazio
    DG = nx.DiGraph()
    
    # Adiciona os dois nós terminais (classes de destino)
    DG.add_node('Class_A', node_type='terminal', class_label='A')
    DG.add_node('Class_B', node_type='terminal', class_label='B')
    
    # FASE 2: CONSTRUÇÃO DOS CAMINHOS E ACUMULAÇÃO DE PESOS
    # Itera sobre cada bag (cada bag sugere um caminho no grafo)
    for bag_name, bag_predicates_dict in bags_result.items():
        
        # 2.1: Obtém o ranking de métricas para este bag

        # predicate_ranking é um DataFrame com colunas ['Predicate', metric_column]
        # Já está ordenado por metric_column (maior → menor)
        predicate_ranking = predicate_ranking_dict[bag_name]
        
        # Extrai lista ordenada de predicados
        ordered_predicates = predicate_ranking['Predicate'].tolist()
        
        # Filtra apenas predicados que existem neste bag específico
        # (nem todos predicados do ranking podem estar presentes no bag)
        ordered_predicates = [p for p in ordered_predicates if p in bag_predicates_dict.keys()]
        
        # Se não há predicados válidos neste bag, pula para o próximo
        if len(ordered_predicates) == 0:
            continue
        
        # 2.2: Cria dicionário de lookup para obter metrica de ranking de cada predicado

        # Isso facilita buscar o valor da métrica pelo nome do predicado
        # Exemplo: ranking_lookup['Ca ka <= 25.5'] = 0.85
        ranking_lookup = dict(zip(predicate_ranking['Predicate'], predicate_ranking[metric_column]))
        
        # 2.3: Constrói arestas entre predicados consecutivos

        # Para cada par consecutivo (pred_current → pred_next) na lista ordenada:
        # - Adiciona os nós se não existirem
        # - Cria aresta com peso = Metrica do predicado de ORIGEM (pred_current)
        # - Se aresta já existe, ACUMULA o peso (soma as Metricas de diferentes bags)
        
        for i in range(len(ordered_predicates) - 1):
            # Predicado atual (origem da aresta)
            pred_current = ordered_predicates[i]
            # Próximo predicado (destino da aresta)
            pred_next = ordered_predicates[i + 1]
            
            # Adiciona nós ao grafo (se já existir, não faz nada)
            DG.add_node(pred_current, node_type='predicate')
            DG.add_node(pred_next, node_type='predicate')
            
            # Obtém a métrica do predicado de ORIGEM
            ranking_value = float(ranking_lookup[pred_current])
            
            # NOVA FEATURE: Multiplicar pela variância explicada se var_exp=True
            if var_exp:
                zone_name = predicate_to_zone.get(pred_current)
                if zone_name and zone_name in pca_info_dict:
                    variance_explained = pca_info_dict[zone_name]['variance_explained']
                    ranking_value = ranking_value * variance_explained
            
            # Verifica se a aresta já existe
            if DG.has_edge(pred_current, pred_next):
                # Se existe, ACUMULA o peso (soma as MIs de diferentes bags)
                DG[pred_current][pred_next]['weight'] += ranking_value
            else:
                # Se não existe, cria a aresta com o peso inicial
                DG.add_edge(pred_current, pred_next, weight=ranking_value, bag=bag_name)
        
        # 2.4: Conecta o ÚLTIMO predicado ao nó terminal

        # O último predicado determina a classe final baseada na maioria
        last_pred = ordered_predicates[-1]
        
        # Garante que o nó do último predicado existe
        # (necessário para bags com apenas 1 predicado, que não entram no loop acima)
        DG.add_node(last_pred, node_type='predicate')
        
        # Obtém o DataFrame de amostras que satisfazem o último predicado
        df_last = bag_predicates_dict[last_pred]
        
        # Conta quantas amostras de cada classe
        class_counts = df_last['Class_Predicted'].value_counts()
        
        # Determina a classe majoritária
        majority_class = class_counts.idxmax()
        
        # Define o nó terminal de destino
        terminal_node = f'Class_{majority_class}'
        
        # Peso da aresta para o terminal = ranking_value do último predicado
        ranking_last_value = float(ranking_lookup[last_pred])
        
        # NOVA FEATURE: Multiplicar pela variância explicada se var_exp=True
        if var_exp:
            zone_name = predicate_to_zone.get(last_pred)
            if zone_name and zone_name in pca_info_dict:
                variance_explained = pca_info_dict[zone_name]['variance_explained']
                ranking_last_value = ranking_last_value * variance_explained
        
        # Acumula ou cria a aresta para o terminal
        if DG.has_edge(last_pred, terminal_node):
            DG[last_pred][terminal_node]['weight'] += ranking_last_value
        else:
            DG.add_edge(last_pred, terminal_node, weight=ranking_last_value, bag=bag_name)

    # FASE 3: IDENTIFICAÇÃO DE ARESTAS BIDIRECIONAIS

    # Quando bags diferentes sugerem ordens opostas (A→B em um, B→A em outro),
    # temos arestas bidirecionais que precisam ser resolvidas
    
    bidirectional_pairs = []  # Lista para armazenar pares bidirecionais
    processed = set()         # Set para evitar processar o mesmo par duas vezes
    
    # Itera sobre todas as arestas do grafo
    for u, v in DG.edges():
        # Verifica se existe a aresta reversa (v → u) E se ainda não processamos este par
        if DG.has_edge(v, u) and (v, u) not in processed:
            # Obtém os pesos de ambas direções
            weight_forward = float(DG[u][v]['weight'])
            weight_reverse = float(DG[v][u]['weight'])
            
            # Armazena informações do par bidirecional
            bidirectional_pairs.append({
                'node_A': u,
                'node_B': v,
                'weight_A_to_B': weight_forward,
                'weight_B_to_A': weight_reverse
            })
            
            # Marca ambas direções como processadas
            processed.add((u, v))
            processed.add((v, u))
    
    print(f"\nTotal de pares bidirecionais encontrados: {len(bidirectional_pairs)}")
    
    # FASE 4: RESOLUÇÃO DE ARESTAS BIDIRECIONAIS

    # Critério: mantém a aresta com MAIOR peso (soma de MIs acumuladas)
    # Em caso de empate: escolha aleatória (controlada por random_state)
    
    n_removed = 0  # Contador de arestas removidas
    
    for pair in bidirectional_pairs:
        u = pair['node_A']
        v = pair['node_B']
        weight_forward = pair['weight_A_to_B']
        weight_reverse = pair['weight_B_to_A']
        
        if weight_forward > weight_reverse:
            # A→B é mais forte: remove B→A
            DG.remove_edge(v, u)
            if show_details:
                print(f"Removida aresta {v} -> {u} (peso {weight_reverse:.4f})")
                print(f"Mantida aresta {u} -> {v} (peso {weight_forward:.4f})\n")
                print("="*70 + "\n")
            n_removed += 1
            
        elif weight_reverse > weight_forward:
            # B→A é mais forte: remove A→B
            DG.remove_edge(u, v)
            if show_details:
                print(f"Removida aresta {u} -> {v} (peso {weight_forward:.4f})")
                print(f"Mantida aresta {v} -> {u} (peso {weight_reverse:.4f})\n")
                print("="*70 + "\n")
            n_removed += 1
            
        else:
            # EMPATE: escolha aleatória
            if np.random.rand() > 0.5:
                DG.remove_edge(v, u)
                if show_details:
                    print(f"Empate! Removida aresta {v} -> {u} (peso {weight_reverse:.4f})")
                    print(f"Mantida aresta {u} -> {v} (peso {weight_forward:.4f})\n")
                    print("="*70 + "\n")
            else:
                DG.remove_edge(u, v)
                if show_details:
                    print(f"Empate! Removida aresta {u} -> {v} (peso {weight_forward:.4f})")
                    print(f"Mantida aresta {v} -> {u} (peso {weight_reverse:.4f})\n")
                    print("="*70 + "\n")
            n_removed += 1

    # FASE 5: RESUMO FINAL
    print(f"\n{'='*70}")
    print("RESUMO DO GRAFO CONSTRUÍDO")
    print(f"{'='*70}")
    print(f"Total de arestas iniciais: {DG.number_of_edges() + n_removed}")
    print(f"Total de arestas removidas por bidirecionalidade: {n_removed}")
    print(f"Arestas bidirecionais restantes: {len(bidirectional_pairs) - n_removed}")
    print(f"Total de nós predicados: {len([n for n, attr in DG.nodes(data=True) if attr['node_type'] == 'predicate'])}")
    print(f"Total de nós terminais: {len([n for n, attr in DG.nodes(data=True) if attr['node_type'] == 'terminal'])}")
    print(f"Métrica utilizada para pesos: {metric_column}")
    if var_exp:
        print(f"Ponderação por variância explicada (var_exp): ATIVADA")
    print()
    
    return DG


import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    # DG = build_predicate_graphv2(
    #     bags_result=all_results_cov[seed]['bags_result'],
    #     predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
    #     metric_column='Covariance',  # ou 'Covariance' se mudar a métrica
    #     random_state=seed,
    #     show_details=True
    # )

    DG = build_predicate_graphv2(
    bags_result=all_results_cov[seed]['bags_result'],
    predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
    metric_column='Covariance',
    random_state=seed,
    show_details=True,
    var_exp=True,                        # ATIVAR
    pca_info_dict=pca_info_dict,         # do aggregate_spectral_zones_pca
    predicates_df=predicates_quantiles[0]
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes

In [ ]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

In [ ]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [ ]:
def extract_predicate_info(predicate_rule):
    """
    Extrai informações de uma regra de predicado.
    
    Parameters
    ----------
    predicate_rule : str
        Regra no formato "zone_name <= threshold" ou "zone_name > threshold"
    
    Returns
    -------
    dict : {'zone': str, 'operator': str, 'threshold': float}
    """
    if '<=' in predicate_rule:
        parts = predicate_rule.split('<=')
        operator = '<='
    elif '>' in predicate_rule:
        parts = predicate_rule.split('>')
        operator = '>'
    else:
        raise ValueError(f"Operador não reconhecido em: {predicate_rule}")
    
    zone_name = parts[0].strip()
    threshold_value = float(parts[1].strip())
    
    return {
        'zone': zone_name,
        'operator': operator,
        'threshold': threshold_value
    }

In [ ]:
def plot_zone_with_threshold(zone_df, threshold_spectrum, zone_name, 
                              class_labels=None, title=None):
    """
    Plota os espectros de uma zona espectral junto com o espectro de threshold.
    
    Parameters
    ----------
    zone_df : pd.DataFrame
        DataFrame com dados espectrais da zona (amostras x variáveis).
    threshold_spectrum : pd.Series
        Espectro de threshold reconstruído.
    zone_name : str
        Nome da zona espectral.
    class_labels : pd.Series, optional
        Labels de classe para colorir as amostras.
    title : str, optional
        Título customizado para o gráfico.
    
    Returns
    -------
    fig : plotly.graph_objects.Figure
        Figura plotly interativa.
    """
    import plotly.graph_objects as go
    import pandas as pd

    fig = go.Figure()
    
    # Converter índice para numérico (energias/comprimentos de onda)
    x_values = pd.to_numeric(zone_df.columns, errors='coerce')
    
    # Plotar espectros das amostras
    if class_labels is not None:
        # Colorir por classe
        colors = {'A': 'gold', 'B': 'blue'}
        for idx, row in zone_df.iterrows():
            class_label = class_labels.iloc[idx] if idx < len(class_labels) else 'Unknown'
            fig.add_trace(go.Scatter(
                x=x_values,
                y=row.values,
                mode='lines',
                line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
                name=f'Class {class_label}',
                showlegend=False,
                hoverinfo='skip'
            ))
    else:
        # Sem coloração por classe
        for idx, row in zone_df.iterrows():
            fig.add_trace(go.Scatter(
                x=x_values,
                y=row.values,
                mode='lines',
                line=dict(color='blue', width=0.5),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Plotar espectro de threshold (destaque)
    fig.add_trace(go.Scatter(
        x=x_values,
        y=threshold_spectrum.values,
        mode='lines',
        line=dict(color='red', width=4, dash='dash'),
        name=f'Threshold Spectrum ({threshold_spectrum.name})'
    ))
    
    # Layout do gráfico
    fig.update_layout(
        title=title or f'Zona Espectral: {zone_name} com Threshold Multivariado',
        xaxis_title='Energia / Comprimento de Onda',
        yaxis_title='Intensidade',
        template='plotly_white',
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    return fig

In [ ]:
n = 2
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[7]['Threshold_Natural'])  # Converter string para float

# Usar pca_info_dict dos dados ORIGINAIS (zone_scores_original_df[1])
pca_info_dict_original = zone_scores_original_df[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]
fig = plot_zone_with_threshold(
    zone_df=zone_df,
    threshold_spectrum=threshold_spectrum,
    zone_name=zone_name,
    class_labels=ycalclass,
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})"
)
fig.show()

In [ ]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='regression',
        metric='mean_abs_diff', 
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    # DG = build_predicate_graphv2(
    #     bags_result=all_results_pert[seed]['bags_result'],
    #     predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
    #     metric_column='Perturbation',  # ou 'Covariance' se mudar a métrica
    #     random_state=seed,
    #     show_details=True
    # )

    DG = build_predicate_graphv2(
    bags_result=all_results_pert[seed]['bags_result'],
    predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
    metric_column='Perturbation',
    random_state=seed,
    show_details=True,
    var_exp=True,                        # ATIVAR
    pca_info_dict=pca_info_dict,         # do aggregate_spectral_zones_pca
    predicates_df=predicates_quantiles[0]
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes

In [ ]:
all_results_pert[0]['pert_results_dict']['Bag_1']

In [ ]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

In [ ]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zone_scores_original_df[0]
)

lrc_summed_df_pert_natural

In [ ]:
# Permutation importance baseado em mudança nas predições do modelo PLS
# Medimos a média da diferença absoluta entre as predições originais e as predições
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Predições base do modelo PLS
baseline_pred = pls_model.predict(Xcalclass_prep)
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_pred = pls_model.predict(X_perm)
        diffs.append(np.mean(np.abs(baseline_pred - perm_pred)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance': importance_list
})
permutation_df.sort_values(by='Permutation_importance', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance', ascending=False)
permutation_unique_df

In [ ]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    #len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'VIP_Score': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_Coefficient' : pad_list(reg_vet_unique_df['Zone'], max_len),
    #'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

#features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

In [ ]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
#rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

In [ ]:
n = 9
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_pert_natural.iloc[7]['Threshold_Natural'])  # Converter string para float

# Usar pca_info_dict dos dados ORIGINAIS (zone_scores_original_df[1])
pca_info_dict_original = zone_scores_original_df[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]
fig = plot_zone_with_threshold(
    zone_df=zone_df,
    threshold_spectrum=threshold_spectrum,
    zone_name=zone_name,
    class_labels=ycalclass,
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})"
)
fig.show()

In [ ]:
n = 9
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[7]['Threshold_Natural'])  # Converter string para float

# Usar pca_info_dict dos dados ORIGINAIS (zone_scores_original_df[1])
pca_info_dict_original = zone_scores_original_df[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]
fig = plot_zone_with_threshold(
    zone_df=zone_df,
    threshold_spectrum=threshold_spectrum,
    zone_name=zone_name,
    class_labels=ycalclass,
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})"
)
fig.show()